In [7]:
Pkg.instantiate()

In [2]:
import Pkg;

Pkg.add("QuantumOptics")

    Updating registry at `~/.julia/registries/General.toml`
     Project No packages added to or removed from `~/projects/TestSQRLab/Project.toml`
    Manifest No packages added to or removed from `~/projects/TestSQRLab/Manifest.toml`


In [3]:
using Pkg
Pkg.build("PyCall")

┌ Warning: The active manifest file has dependencies that were resolved with a different julia version (1.12.3). Unexpected behavior may occur.
└ @ nothing /home/jewoop/projects/TestSQRLab/Manifest.toml:0
    Building Conda ─→ `~/.julia/scratchspaces/44cfe95a-1eb2-52ea-b672-e2afdf69b78f/8f06b0cfa4c514c7b9546756dbae91fcfbc92dc9/build.log`
    Building PyCall → `~/.julia/scratchspaces/44cfe95a-1eb2-52ea-b672-e2afdf69b78f/9816a3826b0ebf49ab4926e2b18842ad8b5c8f04/build.log`


In [ ]:
ENV["MPLBACKEND"] = "agg"
using QuantumOptics
using PyPlot
using LinearAlgebra

# -------------------------------------------------------------
# QCRB Calculation 
# -------------------------------------------------------------
function calculate_qcrb(state, state_perturbed, d_alpha::Float64; tol=1e-10)
    rho_dense_data = dense(state).data
    rho_pert_dense_data = dense(state_perturbed).data
    drho_data = (rho_pert_dense_data - rho_dense_data) / d_alpha
    λ, V = LinearAlgebra.eigen(Hermitian(rho_dense_data))
    
    qfi = 0.0
    n = length(λ)
    for i in 1:n
        for j in 1:n
            if real(λ[i] + λ[j]) > tol
                @views v_i = V[:, i]
                @views v_j = V[:, j]
                drho_ij = dot(v_i, drho_data * v_j)
                qfi += 2 * (abs(drho_ij)^2) / real(λ[i] + λ[j])
            end
        end
    end
    return qfi > 0 ? 1.0 / qfi : Inf
end

# -------------------------------------------------------------
# Stepwise Trotter Expansion (Optimized)
# -------------------------------------------------------------
function get_stepwise_combined_theory_rho(initial_rho::Operator, Gamma_phi_t::Float64, Gamma_cool_t::Float64, Gamma_heat_t::Float64, beta_target::Float64, steps::Int=40)
    b = initial_rho.basis_l
    N_cut = b.N
    
    a_op = QuantumOptics.destroy(b)
    at_op = QuantumOptics.create(b)
    n_op = QuantumOptics.number(b)
    I_op = QuantumOptics.identityoperator(b)
    
    d_beta = beta_target / steps
    d_g_phi = Gamma_phi_t / steps
    d_g_c = Gamma_cool_t / steps
    d_g_h = Gamma_heat_t / steps
    
    # Precompute constant matrices to avoid allocations inside the loop
    D_step = QuantumOptics.displace(b, d_beta)
    D_step_dag = QuantumOptics.dagger(D_step)
    
    M0_c = I_op - (d_g_c / 2.0) * n_op
    M1_c = sqrt(d_g_c) * a_op
    M0_c_dag = QuantumOptics.dagger(M0_c)
    M1_c_dag = QuantumOptics.dagger(M1_c)
    
    M0_h = I_op - (d_g_h / 2.0) * (n_op + I_op)
    M1_h = sqrt(d_g_h) * at_op
    M0_h_dag = QuantumOptics.dagger(M0_h)
    M1_h_dag = QuantumOptics.dagger(M1_h)
    
    # Precompute dephasing array mask for blazing-fast element-wise matrix updates
    dephasing_mask = ones(Float64, N_cut + 1, N_cut + 1)
    if d_g_phi > 0
        for i in 1:(N_cut + 1)
            for j in 1:(N_cut + 1)
                dephasing_mask[i, j] = exp(-0.5 * d_g_phi * ((i - 1) - (j - 1))^2)
            end
        end
    end
    
    rho = deepcopy(initial_rho)
    
    for k in 1:steps
        rho = D_step * rho * D_step_dag
        
        if d_g_phi > 0
            rho.data .*= dephasing_mask  # In-place broadcast multiplication bypasses allocation
        end
        if d_g_c > 0
            rho = M0_c * rho * M0_c_dag + M1_c * rho * M1_c_dag
        end
        if d_g_h > 0
            rho = M0_h * rho * M0_h_dag + M1_h * rho * M1_h_dag
        end
    end
    return rho
end

# -------------------------------------------------------------
# Phase Diagram Plotter (With Precomputed Readouts)
# -------------------------------------------------------------
function plot_multichannel_phase_diagram(; sweep_channel::Symbol=:dephasing, beta_signal::Float64=2.0)
    g_min_sweep = 1e-2
    g_max_sweep = 1e2
    sweep_axis = 10.0 .^ range(log10(g_min_sweep), log10(g_max_sweep), length=100)
    
    n_vals = [0, 1, 2, 3]
    r_vals = [0.0]
    
    base_dephasing = 0.0
    base_cooling = 0.0
    base_heating = 0.0
    
    alphas = collect(0.1:0.01:2.0)
    d_beta = 1e-4
    
    b_spin = SpinBasis(1//2)
    b_motion = FockBasis(50)
    a_op = QuantumOptics.destroy(b_motion)
    
    colors = Dict(0=>"C0", 1=>"C1", 2=>"C2", 3=>"C3")
    linestyles = Dict(0.0=>"-", 0.2=>"--", 0.4=>":")
    curve_data = Dict()
    
    println("==========================================================")
    println(" PRECOMPUTING SDF READOUT OPERATORS (Solves 20min lag)...")
    println("==========================================================")
    g = basisstate(b_spin, 1)
    e = basisstate(b_spin, 2)
    x_plus = normalize(g + e)
    x_minus = normalize(g - e)
    P_plus = x_plus ⊗ dagger(x_plus)
    P_minus = x_minus ⊗ dagger(x_minus)
    
    # Store SDF pulse matrices so we never call displace() inside the loop again
    U_SDF_list = Operator[]
    U_SDF_dag_list = Operator[]
    for a in alphas
        alpha_complex = a * exp(1im * pi/2)
        D_plus = QuantumOptics.displace(b_motion, alpha_complex)
        D_minus = QuantumOptics.displace(b_motion, -alpha_complex)
        U_SDF = (P_plus ⊗ D_plus) + (P_minus ⊗ D_minus)
        push!(U_SDF_list, U_SDF)
        push!(U_SDF_dag_list, dagger(U_SDF))
    end
    
    fig = figure(figsize=(12, 8))
    println(" CALCULATING QCRB PHASE DIAGRAM (TROTTERIZED)")
    
    # Threads.@threads allows Julia to process the n and r loop using multiple CPU cores
    for n in n_vals
        for r in r_vals
            println("Evaluating Fock n=$n, Squeezing r=$r ...")
            qcrb_mins = Float64[]
            
            S_op = QuantumOptics.squeeze(b_motion, r)
            psi_init = S_op * fockstate(b_motion, n)
            rho_init = dm(psi_init)
            
            for g_val in sweep_axis
                G_phi = sweep_channel == :dephasing ? g_val : base_dephasing
                G_cool = sweep_channel == :cooling ? g_val : base_cooling
                G_heat = sweep_channel == :heating ? g_val : base_heating
                
                safe_steps = max(500, Int(ceil(4.0 * g_val * b_motion.N)))
                
                rho_final = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal, safe_steps)
                rho_final_pert = get_stepwise_combined_theory_rho(rho_init, G_phi, G_cool, G_heat, beta_signal + d_beta, safe_steps)
                
                D_shift = displace(b_motion, -expect(a_op, rho_final))
                rho_shift = D_shift * rho_final * dagger(D_shift)
                rho_shift_pert = D_shift * rho_final_pert * dagger(D_shift)
                
                spin_g = basisstate(b_spin, 1)
                sys_state = dm(spin_g) ⊗ rho_shift
                sys_state_pert = dm(spin_g) ⊗ rho_shift_pert
                
                min_qcrb = Inf
                # Super-fast optimized inner loop uses precomputed U_SDF matrices
                for (idx, a) in enumerate(alphas)
                    U_SDF = U_SDF_list[idx]
                    U_SDF_dag = U_SDF_dag_list[idx]
                    
                    out_state = U_SDF * sys_state * U_SDF_dag
                    out_state_pert = U_SDF * sys_state_pert * U_SDF_dag
                    
                    spin_out = ptrace(out_state, 2)
                    spin_out_pert = ptrace(out_state_pert, 2)
                    
                    qcrb = calculate_qcrb(spin_out, spin_out_pert, d_beta)
                    if qcrb < min_qcrb
                        min_qcrb = qcrb
                    end
                end
                push!(qcrb_mins, min_qcrb)
            end
            
            curve_data[(n, r)] = qcrb_mins
            lbl = "n=$n, r=$r"
            plot(sweep_axis, qcrb_mins, color=colors[n], linestyle=linestyles[r], linewidth=2.5, label=lbl)
        end
    end
    
    # -------------------------------------------------------------
    # EXTRACT CROSSOVERS
    # -------------------------------------------------------------
    println("\nExtracting Crossovers...")
    cross_pts_x = Float64[]
    cross_pts_y = Float64[]
    
    for r in r_vals
        for n in 1:maximum(n_vals)
            if haskey(curve_data, (n, r)) && haskey(curve_data, (n-1, r))
                y1 = curve_data[(n, r)]
                y2 = curve_data[(n-1, r)]
                
                for k in 1:(length(sweep_axis)-1)
                    diff1 = y1[k] - y2[k]
                    diff2 = y1[k+1] - y2[k+1]
                    
                    if diff1 * diff2 <= 0.0
                        log_x1, log_x2 = log10(sweep_axis[k]), log10(sweep_axis[k+1])
                        log_y1_A, log_y1_B = log10(y1[k]), log10(y1[k+1])
                        log_y2_A, log_y2_B = log10(y2[k]), log10(y2[k+1])
                        
                        t = (log_y2_A - log_y1_A) / (log_y1_B - log_y1_A - log_y2_B + log_y2_A)
                        
                        cross_x = 10.0^((1-t)*log_x1 + t*log_x2)
                        cross_y = 10.0^((1-t)*log_y1_A + t*log_y1_B)
                        
                        push!(cross_pts_x, cross_x)
                        push!(cross_pts_y, cross_y)
                        
                        println("Crossover (n=$(n) -> n=$(n-1), r=$r) at Dimensionless Noise = $(round(cross_x, digits=4))")
                    end
                end
            end
        end
    end
    
    if length(cross_pts_x) > 0
        scatter(cross_pts_x, cross_pts_y, color="black", s=100, zorder=5, marker="x", label="Crossovers")
    end
    
    xscale("log"); yscale("log")
    xlim(g_min_sweep, g_max_sweep)
    ylim(1e-2, 1e2)
    
    xlabel("Dimensionless Noise (" * String(sweep_channel) * ")", fontsize=12)
    ylabel("min(QCRB)", fontsize=12)
    
    title_str = "QCRB Phase Diagram (Squeezed Fock States)\nSweeping: $(sweep_channel) | Base: Γ_ϕt=$(base_dephasing), Γ_ct=$(base_cooling), Γ_ht=$(base_heating)"
    title(title_str, fontsize=14)
    
    grid(alpha=0.4)
    legend(fontsize=9, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=2)
    tight_layout()
    
    display(fig)
end

plot_multichannel_phase_diagram(sweep_channel=:cooling, beta_signal=2.0)